In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/synthetic_transactions.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['cardholder_id', 'timestamp']).reset_index(drop=True)

print(f"Loaded: {len(df):,} transactions, {df['cardholder_id'].nunique():,} cardholders")
print(f"Fraud: {df['is_fraud'].sum():,} ({df['is_fraud'].mean()*100:.3f}%)")

Loaded: 500,000 transactions, 5,000 cardholders
Fraud: 2,088 (0.418%)


In [2]:
# FEATURE ENGINEERING — REAL TRANSACTION FEATURES
# Everything here was impossible with Kaggle's V1-V28

print("Engineering features...")

# === 1. TIME FEATURES ===
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek  # 0=Monday, 6=Sunday
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['is_night'] = ((df['hour'] >= 0) & (df['hour'] < 6)).astype(int)

# === 2. TIME SINCE LAST TRANSACTION (per cardholder) ===
df['prev_timestamp'] = df.groupby('cardholder_id')['timestamp'].shift(1)
df['time_since_last_txn'] = (df['timestamp'] - df['prev_timestamp']).dt.total_seconds()
df['time_since_last_txn'] = df['time_since_last_txn'].fillna(-1)  # first transaction

# Rapid transactions = suspicious
df['is_rapid'] = (df['time_since_last_txn'].between(0, 300)).astype(int)  # within 5 minutes

print("  Time features done")

# === 3. AMOUNT FEATURES (per cardholder) ===
# Rolling average — what does this cardholder normally spend?
df['ch_avg_amount'] = df.groupby('cardholder_id')['amount'].transform(
    lambda x: x.expanding().mean().shift(1)
)
df['ch_std_amount'] = df.groupby('cardholder_id')['amount'].transform(
    lambda x: x.expanding().std().shift(1)
)

# How unusual is this amount FOR THIS CARDHOLDER?
df['amount_vs_personal_avg'] = (df['amount'] - df['ch_avg_amount']) / (df['ch_std_amount'] + 1e-6)
df['amount_vs_personal_avg'] = df['amount_vs_personal_avg'].fillna(0)

# Amount log transform
df['amount_log'] = np.log1p(df['amount'])

# Is it a round number?
df['is_round_amount'] = (df['amount'] % 10 == 0).astype(int)

print("  Amount features done")

# === 4. VELOCITY FEATURES ===
# How many transactions today for this cardholder?
df['date'] = df['timestamp'].dt.date
df['daily_txn_count'] = df.groupby(['cardholder_id', 'date'])['amount'].transform('count')

# Daily spend for this cardholder
df['daily_spend'] = df.groupby(['cardholder_id', 'date'])['amount'].transform('sum')

# Rolling 7-day transaction count
df['txn_count_7d'] = df.groupby('cardholder_id')['amount'].transform(
    lambda x: x.rolling(7, min_periods=1).count()
)

print("  Velocity features done")

# === 5. MERCHANT / CATEGORY FEATURES ===
# Has this cardholder used this merchant before?
df['merchant_visit_count'] = df.groupby(['cardholder_id', 'merchant_name']).cumcount()
df['is_new_merchant'] = (df['merchant_visit_count'] == 0).astype(int)

# Has this cardholder used this category before?
df['category_visit_count'] = df.groupby(['cardholder_id', 'merchant_category']).cumcount()
df['is_new_category'] = (df['category_visit_count'] == 0).astype(int)

# Category fraud rate (global) — merchant category risk score
cat_fraud_rate = df.groupby('merchant_category')['is_fraud'].mean()
df['category_risk_score'] = df['merchant_category'].map(cat_fraud_rate)

print("  Merchant features done")

# === 6. GEOGRAPHIC FEATURES ===
# Has this cardholder been to this city before?
df['city_visit_count'] = df.groupby(['cardholder_id', 'merchant_city']).cumcount()
df['is_new_city'] = (df['city_visit_count'] == 0).astype(int)

# International when cardholder is normally domestic
ch_intl_rate = df.groupby('cardholder_id')['is_international'].transform(
    lambda x: x.expanding().mean().shift(1)
).fillna(0)
df['intl_deviation'] = df['is_international'].astype(int) - ch_intl_rate

# Different country from last transaction?
df['prev_country'] = df.groupby('cardholder_id')['merchant_country'].shift(1)
df['country_changed'] = (df['merchant_country'] != df['prev_country']).astype(int)
df['country_changed'] = df['country_changed'].fillna(0)

print("  Geographic features done")

# === 7. CHANNEL FEATURES ===
# Online ratio shift — sudden change from in-store to online
df['is_online_int'] = df['is_online'].astype(int)
ch_online_rate = df.groupby('cardholder_id')['is_online_int'].transform(
    lambda x: x.expanding().mean().shift(1)
).fillna(0.5)
df['online_ratio_shift'] = df['is_online_int'] - ch_online_rate

print("  Channel features done")

# === 8. SPENDING ACCELERATION (bust-out detection) ===
# Compare last 3 days spend to previous 7 days average
df['amount_rolling_3d'] = df.groupby('cardholder_id')['amount'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)
df['amount_rolling_7d'] = df.groupby('cardholder_id')['amount'].transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)
df['spending_acceleration'] = (df['amount_rolling_3d'] / (df['amount_rolling_7d'] + 1e-6))

print("  Acceleration features done")

# === CLEAN UP ===
df.drop(columns=['prev_timestamp', 'date', 'prev_country', 'is_online_int'], inplace=True)

# Summary
original_cols = ['transaction_id', 'cardholder_id', 'timestamp', 'merchant_name',
                  'merchant_category', 'mcc', 'merchant_city', 'merchant_state',
                  'merchant_country', 'amount', 'currency', 'is_online',
                  'is_recurring', 'is_international', 'is_fraud', 'fraud_type']
new_features = [col for col in df.columns if col not in original_cols]

print(f"\n{'='*50}")
print(f"FEATURE ENGINEERING COMPLETE")
print(f"{'='*50}")
print(f"Original columns: {len(original_cols)}")
print(f"New features: {len(new_features)}")
print(f"Total columns: {len(df.columns)}")
print(f"\nNew features:")
for f in new_features:
    print(f"  {f}")

Engineering features...
  Time features done
  Amount features done
  Velocity features done
  Merchant features done
  Geographic features done
  Channel features done
  Acceleration features done

FEATURE ENGINEERING COMPLETE
Original columns: 16
New features: 27
Total columns: 43

New features:
  hour
  day_of_week
  is_weekend
  is_night
  time_since_last_txn
  is_rapid
  ch_avg_amount
  ch_std_amount
  amount_vs_personal_avg
  amount_log
  is_round_amount
  daily_txn_count
  daily_spend
  txn_count_7d
  merchant_visit_count
  is_new_merchant
  category_visit_count
  is_new_category
  category_risk_score
  city_visit_count
  is_new_city
  intl_deviation
  country_changed
  online_ratio_shift
  amount_rolling_3d
  amount_rolling_7d
  spending_acceleration


In [3]:
# WHICH FEATURES CATCH FRAUD?
feature_cols = [col for col in new_features if df[col].dtype in ['float64', 'int64', 'int32']]

# Compare fraud vs legitimate for each feature
print(f"FEATURE POWER — FRAUD vs LEGITIMATE")
print(f"{'='*70}")
print(f"{'Feature':<28} {'Legit Mean':>12} {'Fraud Mean':>12} {'Separation':>12}")
print(f"{'-'*70}")

separations = {}
for f in feature_cols:
    legit_mean = df[df['is_fraud']==0][f].mean()
    fraud_mean = df[df['is_fraud']==1][f].mean()
    legit_std = df[df['is_fraud']==0][f].std() + 1e-6
    separation = abs(fraud_mean - legit_mean) / legit_std
    separations[f] = separation
    
    marker = " ***" if separation > 0.5 else " **" if separation > 0.2 else ""
    print(f"{f:<28} {legit_mean:>12.4f} {fraud_mean:>12.4f} {separation:>12.4f}{marker}")

# Top 10 most discriminative features
print(f"\n\nTOP 10 FRAUD-DETECTING FEATURES")
print(f"{'='*50}")
top_features = sorted(separations.items(), key=lambda x: x[1], reverse=True)[:10]
for i, (feat, sep) in enumerate(top_features, 1):
    fraud_mean = df[df['is_fraud']==1][feat].mean()
    legit_mean = df[df['is_fraud']==0][feat].mean()
    direction = "higher" if fraud_mean > legit_mean else "lower"
    print(f"  {i:>2}. {feat:<28} separation: {sep:.4f}  (fraud is {direction})")

FEATURE POWER — FRAUD vs LEGITIMATE
Feature                        Legit Mean   Fraud Mean   Separation
----------------------------------------------------------------------
hour                              12.8535      12.7941       0.0116
day_of_week                        2.9742       2.9344       0.0200
is_weekend                         0.2781       0.2644       0.0307
is_night                           0.0594       0.0771       0.0751
time_since_last_txn           152612.9340   67282.7088       0.5362 ***
is_rapid                           0.0028       0.0268       0.4574 **
ch_avg_amount                     17.8183      42.9676       2.4646 ***
ch_std_amount                     20.0716      86.4193       2.3083 ***
amount_vs_personal_avg           154.7021       8.6745       0.0015
amount_log                         2.5960       5.2702       3.7966 ***
is_round_amount                    0.0063       0.0034       0.0370
daily_txn_count                    1.5705       2.6432    

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score
import mlflow

# Encode categorical features
le_category = LabelEncoder()
df['merchant_category_encoded'] = le_category.fit_transform(df['merchant_category'])

le_country = LabelEncoder()
df['merchant_country_encoded'] = le_country.fit_transform(df['merchant_country'])

# Select features for training
feature_cols = [
    'hour', 'day_of_week', 'is_weekend', 'is_night',
    'time_since_last_txn', 'is_rapid',
    'amount', 'amount_log', 'ch_avg_amount', 'ch_std_amount', 
    'amount_vs_personal_avg', 'is_round_amount',
    'daily_txn_count', 'daily_spend', 'txn_count_7d',
    'merchant_visit_count', 'is_new_merchant',
    'category_visit_count', 'is_new_category', 'category_risk_score',
    'city_visit_count', 'is_new_city',
    'intl_deviation', 'country_changed',
    'online_ratio_shift',
    'amount_rolling_3d', 'amount_rolling_7d', 'spending_acceleration',
    'merchant_category_encoded', 'merchant_country_encoded',
    'is_online', 'is_recurring', 'is_international',
]

# Convert booleans to int
for col in feature_cols:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype(int)

X = df[feature_cols].fillna(0)
y = df['is_fraud']

# Time-based split
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

scale = len(y_train[y_train==0]) / len(y_train[y_train==1])

print(f"Training: {len(X_train):,} transactions, {y_train.sum()} frauds")
print(f"Test: {len(X_test):,} transactions, {y_test.sum()} frauds")
print(f"Features: {len(feature_cols)}")

# Train with MLflow
mlflow.set_experiment("fraud-detection-synthetic")

with mlflow.start_run(run_name="synthetic_xgboost_v1"):
    params = {
        "n_estimators": 300,
        "max_depth": 8,
        "learning_rate": 0.05,
        "scale_pos_weight": scale,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "eval_metric": "aucpr",
        "random_state": 42,
        "n_jobs": -1
    }
    
    mlflow.log_params(params)
    mlflow.log_param("dataset", "synthetic_500k")
    mlflow.log_param("num_features", len(feature_cols))
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    auc_pr = average_precision_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn)
    
    mlflow.log_metrics({"auc_pr": auc_pr, "precision": precision, "recall": recall, "f1_score": f1})
    
    print(f"\n{'='*50}")
    print(f"SYNTHETIC DATA MODEL RESULTS")
    print(f"{'='*50}")
    print(f"AUC-PR:     {auc_pr:.4f}")
    print(f"Precision:  {precision:.4f}  ({precision*100:.1f}% of flags are real fraud)")
    print(f"Recall:     {recall:.4f}  (caught {recall*100:.1f}% of all frauds)")
    print(f"F1 Score:   {f1:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"  True Negatives:  {tn:,}  (legitimate, correctly cleared)")
    print(f"  False Positives: {fp:,}  (legitimate, wrongly flagged)")
    print(f"  False Negatives: {fn:,}  (fraud, MISSED)")
    print(f"  True Positives:  {tp:,}  (fraud, CAUGHT)")
    
    # Feature importance
    importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    print(f"\nTOP 10 FEATURES BY MODEL IMPORTANCE:")
    for feat, imp in importances.head(10).items():
        print(f"  {feat:<30} {imp:.4f}")
    
    print(f"\n\nCOMPARISON: Kaggle vs Synthetic")
    print(f"{'='*50}")
    print(f"{'Metric':<20} {'Kaggle':<15} {'Synthetic':<15}")
    print(f"{'-'*50}")
    print(f"{'Features':<20} {'11':<15} {len(feature_cols):<15}")
    print(f"{'AUC-PR':<20} {'0.8040':<15} {auc_pr:<15.4f}")
    print(f"{'Precision':<20} {'87.7%':<15} {precision*100:<15.1f}%")
    print(f"{'Recall':<20} {'76.0%':<15} {recall*100:<15.1f}%")

Training: 400,000 transactions, 1691 frauds
Test: 100,000 transactions, 397 frauds
Features: 33


2026/03/07 14:55:47 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-synthetic' does not exist. Creating a new experiment.



SYNTHETIC DATA MODEL RESULTS
AUC-PR:     0.9359
Precision:  0.8866  (88.7% of flags are real fraud)
Recall:     0.8866  (caught 88.7% of all frauds)
F1 Score:   0.8866

Confusion Matrix:
  True Negatives:  99,558  (legitimate, correctly cleared)
  False Positives: 45  (legitimate, wrongly flagged)
  False Negatives: 45  (fraud, MISSED)
  True Positives:  352  (fraud, CAUGHT)

TOP 10 FEATURES BY MODEL IMPORTANCE:
  amount_log                     0.4174
  amount                         0.3186
  category_risk_score            0.0287
  daily_txn_count                0.0254
  daily_spend                    0.0229
  txn_count_7d                   0.0160
  category_visit_count           0.0135
  is_online                      0.0131
  is_rapid                       0.0112
  country_changed                0.0111


COMPARISON: Kaggle vs Synthetic
Metric               Kaggle          Synthetic      
--------------------------------------------------
Features             11              33      